# Register Qwen3-VL as a Databricks Model Serving Endpoint (vLLM)

Second attempt at this notebook, built closely against Databricks' own
`serve-custom-llms-starter.py` reference notebook (the one linked at the
end of their [Serve custom LLMs with Custom Model Serving](https://docs.databricks.com/aws/en/machine-learning/model-serving/serve-custom-llms)
doc) rather than reconstructed from prose descriptions — the first attempt
(preserved at `register_vision_endpoint_transformers.ipynb`, a plain
`transformers`-based alternative with no JIT-compiled kernels at all) hit
a chain of environment issues that this real reference resolves several
of directly:

- **Exact version pins that Databricks actually tested**: `vllm==0.11.2`
  (not `0.11.0` — we'd picked the bare minimum for Qwen3-VL support,
  not the version they validated), `transformers==4.57.6`, `mlflow==3.12.0`.
- **`env_pack="databricks_model_serving"` on `mlflow.register_model()`** —
  their comment: *"Custom LLM Serving depends on Serverless Optimized
  Deployments. The endpoint will not work without it."* We were missing
  this entirely in both notebooks; also fixed in the `transformers` one.
- **Must log/register from serverless GPU compute**, enforced by an
  explicit `DATABRICKS_ACCELERATOR` env var check — not classic clusters,
  regardless of the earlier FIPS issue. Confirmed, not just inferred.
- **Local test port must be in the 3000-3999 range** (serverless GPU
  notebooks only allowlist that range) — we'd used 8000.

**One real gap that remains**: their reference notebook serves
`Qwen/Qwen3-4B-Instruct-2507`, a **text-only** model — not a
vision-language one. Everything multimodal (`--limit-mm-per-prompt`, the
image-request validation) is still our own extension beyond their tested
path, not something their reference notebook exercises. So this attempt
fixes every *generic* mistake we made, but the vision-specific code path
is still somewhat uncharted territory.

**Prerequisites:** serverless GPU compute (A10), enforced by the
`DATABRICKS_ACCELERATOR` check below — not optional this time, per
Databricks' explicit warning above.


## Set up the environment (ensure you are running Serverless GPU with an A10 GPU)

In [ ]:
%sh
nvidia-smi

In [ ]:
# Exact versions Databricks' own reference notebook validated for this
# serving path, plus huggingface_hub/pdfplumber for our own download and
# end-to-end validation steps (not part of their reference, needed for
# ours).
%pip install "vllm==0.11.2" "transformers==4.57.6" "openai==2.17.0" "opencv-python-headless==4.12.*" "mlflow==3.12.0" "hf_transfer==0.1.9" "databricks-sdk>=0.102.0" "huggingface_hub>=0.24" pdfplumber
%restart_python

In [ ]:
# Set working directory to local disk (/Workspace doesn't support large files).
import os
import tempfile

workdir = tempfile.mkdtemp()
os.chdir(workdir)

## Configuration

Widgets instead of bare constants (matching this project's other
notebooks) — same values as Databricks' reference otherwise.


In [ ]:
from databricks.sdk.service.serving import ServingModelWorkloadType

dbutils.widgets.text("catalog", "eliao")
dbutils.widgets.text("schema", "wnv_demo")
dbutils.widgets.text("model_name", "qwen3_vl_ocr")
dbutils.widgets.text("hf_model_id", "Qwen/Qwen3-VL-4B-Instruct")
dbutils.widgets.text("endpoint_name", "qwen3-vl-ocr")
# GPU_MEDIUM = 1x A10 (24GB) -- Databricks' own docs list this as the
# default tier for general inference; their reference notebook uses it too.
dbutils.widgets.dropdown("workload_type", "GPU_MEDIUM", ["GPU_SMALL", "GPU_MEDIUM", "GPU_LARGE"])
# Only needed for the end-to-end validation cell at the bottom.
dbutils.widgets.text(
    "test_pdf_volume_path",
    "/Volumes/eliao/wnv_demo/documents/WNV-Outbreak-Communications-Toolkit-2025_508c.pdf",
)
dbutils.widgets.text("test_page", "5")

catalog = dbutils.widgets.get("catalog").strip()
schema = dbutils.widgets.get("schema").strip()
model_name = dbutils.widgets.get("model_name").strip()
hf_model_id = dbutils.widgets.get("hf_model_id").strip()
endpoint_name = dbutils.widgets.get("endpoint_name").strip()
workload_type_str = dbutils.widgets.get("workload_type").strip()

# Hugging Face model artifacts, downloaded below.
ARTIFACTS_PATH = "qwen3vl"
SERVED_MODEL_NAME = "qwen3-vl-ocr"

# vLLM tuning -- same values validated in the first attempt's local test
# before it hit the FlashInfer/curand.h wall (model loading, torch.compile,
# and everything up to that point worked fine with these settings).
DTYPE = "float16"
MAX_MODEL_LEN = 8192
GPU_MEMORY_UTILIZATION = 0.85
# Limits images per prompt to 1 (our use case: one rendered page per
# request). Not present in Databricks' text-only reference -- our own
# addition, validated to parse correctly (JSON syntax, not key=value) in
# the first vLLM attempt. Comment out the flag entirely in the entrypoint
# below if it turns out not to be needed for this vLLM version.
LIMIT_MM_PER_PROMPT = '{"image": 1}'

# Allowlisted ports for Serverless GPU notebooks are 3000-3999. Model
# Serving requires 8080.
LOCAL_PORT = 3080
SERVING_PORT = 8080

uc_model_name = f"{catalog}.{schema}.{model_name}"
WORKLOAD_TYPE = getattr(ServingModelWorkloadType, workload_type_str)
WORKLOAD_SIZE = "Small"
SCALE_TO_ZERO_ENABLED = True

print(f"UC model:       {uc_model_name}")
print(f"Endpoint:       {endpoint_name}")
print(f"Source model:   {hf_model_id}")
print(f"GPU workload:   {workload_type_str}")

## Download the model

In [ ]:
from huggingface_hub import snapshot_download

snapshot_download(repo_id=hf_model_id, local_dir=ARTIFACTS_PATH)
print(f"Downloaded {hf_model_id} to {ARTIFACTS_PATH}")

## Test the model in the notebook

In [ ]:
def entrypoint(port: int) -> str:
    args = [
        "python", "-u", "-m", "vllm.entrypoints.openai.api_server",
        "--model", ARTIFACTS_PATH,
        "--served-model-name", SERVED_MODEL_NAME,
        "--host", "0.0.0.0",
        "--port", str(port),
        "--dtype", DTYPE,
        "--max-model-len", str(MAX_MODEL_LEN),
        "--gpu-memory-utilization", str(GPU_MEMORY_UTILIZATION),
        # Single-quoted: this whole list gets " ".join()-ed and handed to
        # `bash -lc` as one flat string, so without shell quoting the
        # space inside the JSON value splits into two separate argv
        # tokens and bash mangles it.
        "--limit-mm-per-prompt", f"'{LIMIT_MM_PER_PROMPT}'",
        # "--enforce-eager",  # Faster server startup at the cost of inference performance.
    ]
    return " ".join(args)

In [ ]:
import subprocess

# Start server in background.
log = open("process.log", "w")
vllm_proc = subprocess.Popen(
    ["bash", "-lc", entrypoint(LOCAL_PORT)],
    stdout=log,
    stderr=subprocess.STDOUT,
    text=True,
    start_new_session=True,
)

In [ ]:
import time
import urllib.request

# Poll until the server is ready (or fails) rather than a blocking shell
# tail -- keeps this runnable as a single notebook pass with a clear
# failure instead of a cell that hangs forever on error.
ready = False
for _ in range(120):  # up to ~10 min for first-time weight load
    if vllm_proc.poll() is not None:
        break  # process exited early -- something is wrong
    try:
        urllib.request.urlopen(f"http://localhost:{LOCAL_PORT}/v1/models", timeout=2)
        ready = True
        break
    except Exception:
        time.sleep(5)

if not ready:
    with open("process.log") as f:
        output = f.read()
    raise RuntimeError(f"vLLM server did not become ready. Full log:\n{output}")

print("Local vLLM server is up.")

In [ ]:
import requests

resp = requests.post(
    f"http://localhost:{LOCAL_PORT}/invocations",
    json={"messages": [{"role": "user", "content": "Reply with exactly: OK"}]},
)
print(resp.json()["choices"][0]["message"]["content"])

Stop the local server before continuing.

In [ ]:
vllm_proc.terminate()
vllm_proc.wait(timeout=30)
print("Local vLLM server stopped.")

## Log the model with our custom entrypoint

In [ ]:
import mlflow
from mlflow.pyfunc.model import ChatCompletionResponse, ChatModel

mlflow.set_registry_uri("databricks-uc")


# Required placeholder. Serving runs the entrypoint, not python_model.predict.
class LLMModel(ChatModel):
    def predict(self, context, messages, params):
        return ChatCompletionResponse.from_dict({"choices": []})


# Must log+register from serverless GPU compute -- otherwise the model is
# packaged with CPU dependencies and the GPU serving endpoint fails to
# start.
if not os.environ.get("DATABRICKS_ACCELERATOR"):
    raise RuntimeError(
        "This model MUST be logged+registered from a serverless GPU runtime, "
        "otherwise the correct dependencies will not be packaged for serving."
    )

model_info = mlflow.pyfunc.log_model(
    name=SERVED_MODEL_NAME,
    python_model=LLMModel(),
    artifacts={"model_dir": ARTIFACTS_PATH},
    metadata={
        "task": "llm/v1/chat",
        "entrypoint": entrypoint(SERVING_PORT),
    },
    extra_pip_requirements=["mlflow==3.12.0"],
)
print(f"Logged: {model_info.model_uri}")

## Register the model to Unity Catalog

In [ ]:
# env_pack is required. Custom LLM Serving depends on Serverless Optimized
# Deployments. The endpoint will not work without it.
# https://docs.databricks.com/aws/en/machine-learning/model-serving/serverless-optimized-deployments
model_version = mlflow.register_model(
    model_info.model_uri, uc_model_name, env_pack="databricks_model_serving"
)
print(f"Registered: {uc_model_name} version {model_version.version}")

## Create an endpoint with the model

`scale_to_zero_enabled=True` — matches the design decision that this
endpoint should cost nothing between ingestion runs (the ingestion job is
scheduled/manually-triggered, not continuous).


In [ ]:
from datetime import timedelta

from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import EndpointCoreConfigInput, ServedEntityInput

config = EndpointCoreConfigInput(
    served_entities=[
        ServedEntityInput(
            entity_name=uc_model_name,
            entity_version=str(model_version.version),
            workload_type=WORKLOAD_TYPE,
            workload_size=WORKLOAD_SIZE,
            scale_to_zero_enabled=SCALE_TO_ZERO_ENABLED,
        )
    ]
)

w = WorkspaceClient()
print(f"Creating endpoint '{endpoint_name}' -- can take 10+ minutes for a first deploy.")
w.serving_endpoints.create_and_wait(name=endpoint_name, config=config, timeout=timedelta(minutes=45))
print("Endpoint ready.")

## Validate end-to-end

Same request shape validated in the OCR spike test
(`notebooks/rag_ocr_spike.ipynb`) — confirms this endpoint is a drop-in
for `WNV_VISION_ENDPOINT` with no other code changes needed. Not part of
Databricks' reference notebook (theirs is text-only) — this is the part
that actually tests the vision-specific path.


In [ ]:
import base64
import io
import json
from urllib import request as urlrequest

import pdfplumber

test_pdf_path = dbutils.widgets.get("test_pdf_volume_path").strip()
test_page_num = int(dbutils.widgets.get("test_page").strip())

context = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
databricks_host = context.apiUrl().get().removeprefix("https://").removesuffix("/")
databricks_token = context.apiToken().get()

with pdfplumber.open(test_pdf_path) as pdf:
    page = pdf.pages[test_page_num - 1]
    image = page.to_image(resolution=300)
    buf = io.BytesIO()
    image.original.save(buf, format="PNG")
    image_bytes = buf.getvalue()

b64 = base64.b64encode(image_bytes).decode()
payload = {
    "messages": [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "Extract all text from this document page as clean markdown."},
                {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{b64}"}},
            ],
        }
    ],
    "max_tokens": 2000,
    "temperature": 0.0,
}
url = f"https://{databricks_host}/serving-endpoints/{endpoint_name}/invocations"
req = urlrequest.Request(
    url,
    data=json.dumps(payload).encode(),
    headers={
        "Authorization": f"Bearer {databricks_token}",
        "Content-Type": "application/json",
    },
    method="POST",
)
with urlrequest.urlopen(req, timeout=120) as resp:
    result = json.loads(resp.read().decode())

print(result["choices"][0]["message"]["content"])

## Done

Set `WNV_VISION_ENDPOINT` to the value of `endpoint_name` above (default
`qwen3-vl-ocr`) wherever the ingestion pipeline configures it. The endpoint
is scale-to-zero, so the first call after being idle will be slow (model
reload) — expected, not a bug.
